# Module 2.3: Seq2Seq 序列到序列架构

## 1. 本章概览 (Overview)

### 📚 学习目标

1. **编码器-解码器架构**：理解 Seq2Seq 的基本结构
2. **注意力机制**：解决长序列信息瓶颈问题
3. **完整训练流程**：实现机器翻译任务
4. **Teacher Forcing**：理解训练技巧

### 🎯 核心问题

- 如何将一个序列转换为另一个序列？
- 为什么需要注意力机制？
- 如何训练 Seq2Seq 模型？

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 口语化用户描述如何改写为标准工单？→ Seq2Seq 的序列转换能力
- 生成的回复质量不稳定怎么调？→ 解码策略（贪心、Beam Search）的选择

### 🗺️ 应用场景

- **机器翻译**：英语 → 中文
- **文本摘要**：长文本 → 摘要
- **对话系统**：问题 → 回答
- **代码生成**：描述 → 代码

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. 动机与背景 (Motivation)

### 序列到序列任务的挑战

传统的神经网络处理**固定长度**的输入和输出，但很多任务需要处理**可变长度**的序列：

- 输入和输出长度不同
- 输入和输出的对应关系复杂
- 需要理解整个输入序列才能生成输出

### Seq2Seq 的解决方案

**编码器-解码器 (Encoder-Decoder)** 是一种将输入序列先压缩编码、再逐步解码生成输出序列的神经网络架构。在本章场景中，它是 Seq2Seq 模型的核心结构，用于处理输入输出长度不同的序列转换任务（如机器翻译）。

**编码器-解码器架构**：

```
输入序列 → [编码器] → 上下文向量 → [解码器] → 输出序列
```

- **编码器**：将输入序列压缩成固定长度的上下文向量
- **解码器**：从上下文向量生成输出序列

### 一个简单的例子

机器翻译："Hello" → "你好"

1. 编码器读取 "Hello"，生成上下文向量
2. 解码器从上下文向量生成 "你好"

## 3. 理论基础 (Theory)

### 3.1 编码器 (Encoder)

**作用**：将输入序列编码成上下文向量

**结构**：
$$h_t = \text{LSTM}(x_t, h_{t-1})$$
$$\text{context} = h_T$$

其中：
- $x_t$: 输入序列的第 $t$ 个词
- $h_t$: 第 $t$ 步的隐藏状态
- $h_T$: 最后一个隐藏状态（上下文向量）

### 3.2 解码器 (Decoder)

**作用**：从上下文向量生成输出序列

**结构**：
$$s_t = \text{LSTM}(y_{t-1}, s_{t-1})$$
$$p(y_t) = \text{softmax}(W_s s_t)$$

其中：
- $s_0 = h_T$ (初始状态是编码器的最后状态)
- $y_{t-1}$: 上一步生成的词
- $s_t$: 解码器的隐藏状态
- $p(y_t)$: 下一个词的概率分布

In [ ]:
# 🔬 Micro Practice: Implement Encoder
# Goal: Build LSTM encoder that processes input sequence
# Expected outcome: Encoder produces context vector

class Encoder(nn.Module):
    """
    LSTM Encoder for Seq2Seq
    """
    
    def __init__(self, input_size, embedding_dim, hidden_dim, num_layers=1):
        """
        Initialize encoder
        
        Args:
            input_size: Vocabulary size
            embedding_dim: Embedding dimension
            hidden_dim: Hidden state dimension
            num_layers: Number of LSTM layers
        """
        super(Encoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(input_size, embedding_dim)
        
        # LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           batch_first=True)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input sequence (batch_size, seq_len)
        
        Returns:
            outputs: All hidden states (batch_size, seq_len, hidden_dim)
            hidden: Final hidden state
            cell: Final cell state
        """
        # Embed input
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        
        # Pass through LSTM
        outputs, (hidden, cell) = self.lstm(embedded)
        
        return outputs, hidden, cell

# Test encoder
vocab_size = 100
embedding_dim = 32
hidden_dim = 64

encoder = Encoder(vocab_size, embedding_dim, hidden_dim).to(device)

# Create dummy input
batch_size = 2
seq_len = 5
x = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)

outputs, hidden, cell = encoder(x)

print("Encoder Test:")
print(f"Input shape: {x.shape}")
print(f"Outputs shape: {outputs.shape}")
print(f"Hidden shape: {hidden.shape}")
print(f"Cell shape: {cell.shape}")
print("✓ Encoder 实现成功！")

In [ ]:
# 🔬 Micro Practice: Implement Decoder
# Goal: Build LSTM decoder that generates output sequence
# Expected outcome: Decoder generates predictions from context

class Decoder(nn.Module):
    """
    LSTM Decoder for Seq2Seq
    """
    
    def __init__(self, output_size, embedding_dim, hidden_dim, num_layers=1):
        """
        Initialize decoder
        
        Args:
            output_size: Vocabulary size
            embedding_dim: Embedding dimension
            hidden_dim: Hidden state dimension
            num_layers: Number of LSTM layers
        """
        super(Decoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(output_size, embedding_dim)
        
        # LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, output_size)
    
    def forward(self, x, hidden, cell):
        """
        Forward pass (one step)
        
        Args:
            x: Input token (batch_size, 1)
            hidden: Hidden state from encoder/previous step
            cell: Cell state from encoder/previous step
        
        Returns:
            output: Predictions (batch_size, output_size)
            hidden: Updated hidden state
            cell: Updated cell state
        """
        # Embed input
        embedded = self.embedding(x)  # (batch_size, 1, embedding_dim)
        
        # Pass through LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        
        # Generate predictions
        output = self.fc(lstm_out.squeeze(1))  # (batch_size, output_size)
        
        return output, hidden, cell

# Test decoder
decoder = Decoder(vocab_size, embedding_dim, hidden_dim).to(device)

# Use encoder's final state as initial state
x_dec = torch.randint(0, vocab_size, (batch_size, 1)).to(device)
output, hidden_new, cell_new = decoder(x_dec, hidden, cell)

print("\nDecoder Test:")
print(f"Input shape: {x_dec.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden shape: {hidden_new.shape}")
print(f"Cell shape: {cell_new.shape}")
print("✓ Decoder 实现成功！")

In [ ]:
# 🔬 Micro Practice: Complete Seq2Seq Model
# Goal: Combine encoder and decoder into complete model
# Expected outcome: End-to-end sequence generation

class Seq2Seq(nn.Module):
    """
    Complete Seq2Seq model with encoder and decoder
    """
    
    def __init__(self, encoder, decoder, device):
        """
        Initialize Seq2Seq model
        
        Args:
            encoder: Encoder module
            decoder: Decoder module
            device: Device to run on
        """
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Forward pass
        
        Args:
            src: Source sequence (batch_size, src_len)
            trg: Target sequence (batch_size, trg_len)
            teacher_forcing_ratio: Probability of using teacher forcing
        
        Returns:
            outputs: Predictions for each time step (batch_size, trg_len, vocab_size)
        """
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        
        # Store outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Encode source sequence
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # First input to decoder is <SOS> token (assume it's index 1)
        input = trg[:, 0].unsqueeze(1)
        
        # Decode step by step
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            
            # Teacher forcing: use ground truth as next input
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input = trg[:, t].unsqueeze(1) if teacher_force else top1
        
        return outputs

# Create complete model
seq2seq = Seq2Seq(encoder, decoder, device).to(device)

# Test complete model
src = torch.randint(0, vocab_size, (batch_size, 5)).to(device)
trg = torch.randint(0, vocab_size, (batch_size, 7)).to(device)

outputs = seq2seq(src, trg, teacher_forcing_ratio=1.0)

print("\nSeq2Seq Model Test:")
print(f"Source shape: {src.shape}")
print(f"Target shape: {trg.shape}")
print(f"Outputs shape: {outputs.shape}")
print("✓ 完整 Seq2Seq 模型实现成功！")

### 3.3 注意力机制 (Attention Mechanism)

**问题**：基础 Seq2Seq 的信息瓶颈

在基础 Seq2Seq 中，编码器必须将整个输入序列压缩成一个固定长度的上下文向量。对于长序列，这会导致信息丢失。

**解决方案**：注意力机制

注意力机制允许解码器在每一步都能访问编码器的所有隐藏状态，并动态地关注最相关的部分。

**数学表示**：

1. **计算注意力分数**：
$$e_{t,i} = \text{score}(s_{t-1}, h_i)$$

2. **归一化得到注意力权重**：
$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$$

3. **计算上下文向量**：
$$c_t = \sum_i \alpha_{t,i} h_i$$

4. **结合上下文生成输出**：
$$s_t = \text{LSTM}([y_{t-1}; c_t], s_{t-1})$$

其中：
- $s_{t-1}$: 解码器上一步的隐藏状态
- $h_i$: 编码器第 $i$ 步的隐藏状态
- $\alpha_{t,i}$: 注意力权重（解码器在第 $t$ 步对编码器第 $i$ 步的关注度）
- $c_t$: 上下文向量（编码器隐藏状态的加权和）

In [ ]:
# 🔬 Micro Practice: Implement Attention Mechanism
# Goal: Build attention layer that computes context vector
# Expected outcome: Attention weights and context vector

class Attention(nn.Module):
    """
    Bahdanau Attention Mechanism
    """
    
    def __init__(self, hidden_dim):
        """
        Initialize attention
        
        Args:
            hidden_dim: Hidden state dimension
        """
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        """
        Compute attention weights and context vector
        
        Args:
            hidden: Decoder hidden state (batch_size, hidden_dim)
            encoder_outputs: All encoder hidden states (batch_size, src_len, hidden_dim)
        
        Returns:
            context: Context vector (batch_size, hidden_dim)
            attention_weights: Attention weights (batch_size, src_len)
        """
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # Repeat decoder hidden state src_len times
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        
        # Concatenate decoder hidden and encoder outputs
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        
        # Compute attention scores
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)
        
        # Normalize to get attention weights
        attention_weights = torch.softmax(attention, dim=1)
        
        # Compute context vector as weighted sum
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)  # (batch_size, hidden_dim)
        
        return context, attention_weights

# Test attention
attention = Attention(hidden_dim).to(device)

# Use encoder outputs from previous test
hidden_test = hidden.squeeze(0)  # (batch_size, hidden_dim)
context, attn_weights = attention(hidden_test, outputs)

print("\nAttention Test:")
print(f"Hidden shape: {hidden_test.shape}")
print(f"Encoder outputs shape: {outputs.shape}")
print(f"Context shape: {context.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention weights sum: {attn_weights.sum(dim=1)}")  # Should be ~1.0
print("✓ 注意力机制实现成功！")

In [ ]:
# 🔬 Micro Practice: Attention-Enhanced Decoder
# Goal: Integrate attention into decoder
# Expected outcome: Decoder uses attention to focus on relevant parts

class AttentionDecoder(nn.Module):
    """
    LSTM Decoder with Attention
    """
    
    def __init__(self, output_size, embedding_dim, hidden_dim, attention, num_layers=1):
        """
        Initialize attention decoder
        
        Args:
            output_size: Vocabulary size
            embedding_dim: Embedding dimension
            hidden_dim: Hidden state dimension
            attention: Attention module
            num_layers: Number of LSTM layers
        """
        super(AttentionDecoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.attention = attention
        
        # Embedding layer
        self.embedding = nn.Embedding(output_size, embedding_dim)
        
        # LSTM layer (input is embedding + context)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, num_layers, 
                           batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, output_size)
    
    def forward(self, x, hidden, cell, encoder_outputs):
        """
        Forward pass with attention
        
        Args:
            x: Input token (batch_size, 1)
            hidden: Hidden state (num_layers, batch_size, hidden_dim)
            cell: Cell state (num_layers, batch_size, hidden_dim)
            encoder_outputs: All encoder hidden states (batch_size, src_len, hidden_dim)
        
        Returns:
            output: Predictions (batch_size, output_size)
            hidden: Updated hidden state
            cell: Updated cell state
            attention_weights: Attention weights (batch_size, src_len)
        """
        # Embed input
        embedded = self.embedding(x)  # (batch_size, 1, embedding_dim)
        
        # Compute attention
        context, attention_weights = self.attention(hidden[-1], encoder_outputs)
        
        # Concatenate embedding and context
        context = context.unsqueeze(1)  # (batch_size, 1, hidden_dim)
        lstm_input = torch.cat((embedded, context), dim=2)
        
        # Pass through LSTM
        lstm_out, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Generate predictions
        output = self.fc(lstm_out.squeeze(1))  # (batch_size, output_size)
        
        return output, hidden, cell, attention_weights

# Test attention decoder
attn_decoder = AttentionDecoder(vocab_size, embedding_dim, hidden_dim, attention).to(device)

x_dec = torch.randint(0, vocab_size, (batch_size, 1)).to(device)
output, hidden_new, cell_new, attn_weights = attn_decoder(x_dec, hidden, cell, outputs)

print("\nAttention Decoder Test:")
print(f"Input shape: {x_dec.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Attention weights (first sample): {attn_weights[0].detach().cpu().numpy()}")
print("✓ 注意力解码器实现成功！")

In [ ]:
# 🔬 Micro Practice: Visualize Attention Weights
# Goal: Create heatmap showing attention alignment
# Expected outcome: Visual representation of attention

def visualize_attention(src_tokens, trg_tokens, attention_weights):
    """
    Visualize attention weights as heatmap
    
    Args:
        src_tokens: Source sequence tokens (list of strings)
        trg_tokens: Target sequence tokens (list of strings)
        attention_weights: Attention weights (trg_len, src_len)
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    im = ax.imshow(attention_weights, cmap='Blues', aspect='auto')
    
    # Set ticks and labels
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right')
    ax.set_yticklabels(trg_tokens)
    
    # Add colorbar
    plt.colorbar(im, ax=ax)
    
    # Labels
    ax.set_xlabel('Source Sequence', fontsize=12)
    ax.set_ylabel('Target Sequence', fontsize=12)
    ax.set_title('Attention Weights Heatmap', fontsize=14, weight='bold')
    
    # Add values in cells
    for i in range(len(trg_tokens)):
        for j in range(len(src_tokens)):
            text = ax.text(j, i, f'{attention_weights[i, j]:.2f}',
                          ha="center", va="center", color="black", fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Create example visualization
src_example = ['I', 'love', 'deep', 'learning']
trg_example = ['我', '爱', '深度', '学习']

# Generate dummy attention weights (normally from model)
np.random.seed(42)
attn_example = np.random.rand(len(trg_example), len(src_example))
attn_example = attn_example / attn_example.sum(axis=1, keepdims=True)  # Normalize

print("Attention Visualization Example:")
print("Source:", src_example)
print("Target:", trg_example)
visualize_attention(src_example, trg_example, attn_example)
print("✓ 注意力可视化完成！")

## 4. 实现细节 (Implementation)

### 4.1 准备数据集

我们使用一个简单的数字序列翻译任务作为演示：
- 输入：数字序列（如 "1 2 3"）
- 输出：反转的序列（如 "3 2 1"）

这个任务足够简单，可以快速验证模型是否工作，同时也展示了 Seq2Seq 的核心概念。

In [ ]:
# 🔬 Micro Practice: Create Toy Dataset
# Goal: Generate simple sequence reversal dataset
# Expected outcome: Training data for Seq2Seq

class ReverseDataset:
    """
    Simple dataset for sequence reversal task
    """
    
    def __init__(self, num_samples=1000, max_len=10):
        """
        Generate dataset
        
        Args:
            num_samples: Number of training samples
            max_len: Maximum sequence length
        """
        self.num_samples = num_samples
        self.max_len = max_len
        
        # Special tokens
        self.PAD_token = 0
        self.SOS_token = 1
        self.EOS_token = 2
        
        # Vocabulary: 0=PAD, 1=SOS, 2=EOS, 3-12=digits 0-9
        self.vocab_size = 13
        
        # Generate data
        self.data = []
        for _ in range(num_samples):
            # Random sequence length
            seq_len = random.randint(3, max_len)
            
            # Generate random sequence (digits 3-12 represent 0-9)
            src = [random.randint(3, 12) for _ in range(seq_len)]
            
            # Target is reversed sequence with SOS and EOS
            trg = [self.SOS_token] + src[::-1] + [self.EOS_token]
            
            self.data.append((src, trg))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def collate_fn(self, batch):
        """
        Collate function for DataLoader
        Pads sequences to same length in batch
        """
        src_batch, trg_batch = zip(*batch)
        
        # Find max lengths
        src_max_len = max(len(s) for s in src_batch)
        trg_max_len = max(len(t) for t in trg_batch)
        
        # 记录原始序列长度（用于 pack_padded_sequence 等操作）
        src_lengths = [len(s) for s in src_batch]
        trg_lengths = [len(t) for t in trg_batch]
        
        # Pad sequences
        src_padded = []
        trg_padded = []
        
        for src, trg in zip(src_batch, trg_batch):
            src_padded.append(src + [self.PAD_token] * (src_max_len - len(src)))
            trg_padded.append(trg + [self.PAD_token] * (trg_max_len - len(trg)))
        
        return (torch.LongTensor(src_padded), torch.LongTensor(trg_padded),
                torch.LongTensor(src_lengths), torch.LongTensor(trg_lengths))

# Create dataset
dataset = ReverseDataset(num_samples=1000, max_len=8)

# Create dataloader
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, 
                       collate_fn=dataset.collate_fn)

# Show examples
print("Dataset Examples:")
for i in range(3):
    src, trg = dataset[i]
    print(f"Source: {src}")
    print(f"Target: {trg}")
    print()

print(f"✓ 数据集创建成功！共 {len(dataset)} 个样本")
print(f"✓ 词汇表大小: {dataset.vocab_size}")

### 4.2 训练循环

**教师强制 (Teacher Forcing)** 是一种序列模型的训练技巧：在解码器的每一步，将上一步的**真实目标词**（而非模型自身的预测）作为输入。在本章场景中，它用于加速 Seq2Seq 模型的训练收敛并提高训练稳定性。

**Teacher Forcing**：在训练时，我们使用真实的目标序列作为解码器的输入，而不是使用模型的预测。这可以加速训练并提高稳定性。

**训练步骤**：
1. 前向传播：通过编码器和解码器
2. 计算损失：交叉熵损失
3. 反向传播：计算梯度
4. 更新参数：使用优化器

In [ ]:
# 🔬 Micro Practice: Training Loop
# Goal: Train Seq2Seq model on reversal task
# Expected outcome: Model learns to reverse sequences

# Initialize model components
# Note: Re-initialize with correct vocab_size from dataset
vocab_size = dataset.vocab_size
embedding_dim = 32
hidden_dim = 64

encoder = Encoder(vocab_size, embedding_dim, hidden_dim).to(device)
attention_module = Attention(hidden_dim).to(device)
decoder = AttentionDecoder(vocab_size, embedding_dim, hidden_dim, attention_module).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=dataset.PAD_token)
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(params, lr=0.001)

def train_epoch(encoder, decoder, dataloader, optimizer, criterion, device):
    """Train for one epoch"""
    encoder.train()
    decoder.train()
    
    total_loss = 0
    for src, trg, src_lengths, trg_lengths in dataloader:
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        # Encode
        encoder_outputs, hidden, cell = encoder(src)
        
        # Decode with teacher forcing
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        
        loss = 0
        input_token = trg[:, 0].unsqueeze(1)  # SOS token
        
        for t in range(1, trg_len):
            output, hidden, cell, _ = decoder(input_token, hidden, cell, encoder_outputs)
            loss += criterion(output, trg[:, t])
            input_token = trg[:, t].unsqueeze(1)  # Teacher forcing
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

# Train model
num_epochs = 20
losses = []

print("Training Seq2Seq Model...")
print("=" * 50)

for epoch in range(num_epochs):
    loss = train_epoch(encoder, decoder, dataloader, optimizer, criterion, device)
    losses.append(loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

print("=" * 50)
print("✓ 训练完成！")

# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.show()

### 4.3 评估和推理

在推理时，我们不使用 teacher forcing，而是使用模型自己的预测作为下一步的输入。

In [ ]:
# 🔬 Micro Practice: Inference and Evaluation
# Goal: Test trained model on new sequences
# Expected outcome: Model correctly reverses sequences

def translate(encoder, decoder, src, max_len=20):
    """
    Translate a source sequence
    
    Args:
        encoder: Trained encoder
        decoder: Trained decoder
        src: Source sequence (1, src_len)
        max_len: Maximum output length
    
    Returns:
        output_tokens: Generated sequence
        attention_weights: Attention weights for visualization
    """
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # Encode
        encoder_outputs, hidden, cell = encoder(src)
        
        # Start with SOS token
        input_token = torch.LongTensor([[dataset.SOS_token]]).to(device)
        
        output_tokens = []
        attention_weights_list = []
        
        for _ in range(max_len):
            output, hidden, cell, attn_weights = decoder(input_token, hidden, cell, encoder_outputs)
            
            # Get predicted token
            predicted_token = output.argmax(1).item()
            output_tokens.append(predicted_token)
            attention_weights_list.append(attn_weights.cpu().numpy()[0])
            
            # Stop if EOS token
            if predicted_token == dataset.EOS_token:
                break
            
            # Use prediction as next input
            input_token = torch.LongTensor([[predicted_token]]).to(device)
    
    return output_tokens, np.array(attention_weights_list)

# Test on examples
print("Evaluation Results:")
print("=" * 60)

for i in range(5):
    src, trg = dataset[i]
    src_tensor = torch.LongTensor([src]).to(device)
    
    output_tokens, attn_weights = translate(encoder, decoder, src_tensor)
    
    # Remove SOS and EOS tokens from target
    trg_clean = [t for t in trg if t not in [dataset.SOS_token, dataset.EOS_token, dataset.PAD_token]]
    output_clean = [t for t in output_tokens if t not in [dataset.EOS_token, dataset.PAD_token]]
    
    # Convert to readable format (subtract 3 to get actual digits)
    src_readable = [t-3 for t in src]
    trg_readable = [t-3 for t in trg_clean]
    output_readable = [t-3 for t in output_clean]
    
    correct = "✓" if output_clean == trg_clean else "✗"
    
    print(f"Example {i+1}: {correct}")
    print(f"  Source:    {src_readable}")
    print(f"  Expected:  {trg_readable}")
    print(f"  Predicted: {output_readable}")
    print()

print("=" * 60)
print("✓ 评估完成！")

### 4.4 Beam Search 与 BLEU 评估

**贪婪解码的局限**：贪婪解码每步取概率最高的词，但局部最优不等于全局最优。例如第一步选了概率稍低但后续路径得分更高的词，整体序列可能更好。

**Beam Search**：保留 $k$ 个最佳候选序列（beam），在每步扩展所有候选后保留得分最高的 $k$ 条路径。最终选择总得分最高的序列。

**BLEU (Bilingual Evaluation Understudy)**：机器翻译标准评估指标，通过比较生成序列与参考序列的 n-gram 重叠度来衡量质量：

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

其中 $p_n$ 是 n-gram 精确率，$BP$ 是长度惩罚因子。

In [ ]:
# 🔬 Micro Practice: Beam Search 解码与 BLEU 评估
# Goal: 实现 Beam Search 和 BLEU 分数计算
# Expected outcome: 理解 Beam Search 相比贪婪解码的改进，掌握 BLEU 评估

from collections import Counter

# ============================================================
# 1. Beam Search 解码
# ============================================================

def beam_search_decode(encoder, decoder, src, beam_size=3, max_len=20):
    """
    Beam Search 解码：保留 beam_size 条最优候选路径
    
    Args:
        encoder: 训练好的编码器
        decoder: 训练好的解码器（带注意力）
        src: 源序列 (1, src_len)
        beam_size: beam 宽度
        max_len: 最大生成长度
    
    Returns:
        best_seq: 最优序列的 token 列表
        best_score: 最优序列的得分
    """
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # 编码源序列
        encoder_outputs, hidden, cell = encoder(src)
        
        # 初始化 beam：每条 beam 记录 (token序列, 累积log概率, hidden, cell)
        beams = [([dataset.SOS_token], 0.0, hidden, cell)]
        completed = []  # 已结束的序列
        
        for _ in range(max_len):
            candidates = []
            
            for seq, score, h, c in beams:
                # 如果序列已结束，直接加入 completed
                if seq[-1] == dataset.EOS_token:
                    completed.append((seq, score))
                    continue
                
                # 解码一步
                input_token = torch.LongTensor([[seq[-1]]]).to(src.device)
                output, h_new, c_new, _ = decoder(input_token, h, c, encoder_outputs)
                log_probs = torch.log_softmax(output, dim=-1)[0]
                
                # 取 top-k 候选
                top_log_probs, top_indices = torch.topk(log_probs, beam_size)
                
                for log_p, idx in zip(top_log_probs, top_indices):
                    new_seq = seq + [idx.item()]
                    new_score = score + log_p.item()
                    candidates.append((new_seq, new_score, h_new, c_new))
            
            if not candidates:
                break
            
            # 保留得分最高的 beam_size 条路径
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:beam_size]
            
            # 如果所有 beam 都已结束
            if all(b[0][-1] == dataset.EOS_token for b in beams):
                completed.extend([(b[0], b[1]) for b in beams])
                break
        
        # 将未结束的 beam 也加入候选
        completed.extend([(b[0], b[1]) for b in beams])
        
        # 按长度归一化得分，选最优
        scored = [(seq, score / len(seq)) for seq, score in completed]
        scored.sort(key=lambda x: x[1], reverse=True)
        
        best_seq, best_score = scored[0]
        return best_seq, best_score


# ============================================================
# 2. BLEU 分数计算
# ============================================================

def compute_bleu(reference, candidate, max_n=4):
    """
    计算 BLEU 分数（简化版，单条参考）
    
    Args:
        reference: 参考序列（token 列表）
        candidate: 候选序列（token 列表）
        max_n: 最大 n-gram 阶数
    Returns:
        bleu: BLEU 分数 (0~1)
    """
    import math
    
    # 计算各阶 n-gram 精确率
    precisions = []
    for n in range(1, max_n + 1):
        # 提取 n-grams
        ref_ngrams = Counter([tuple(reference[i:i+n]) for i in range(len(reference)-n+1)])
        cand_ngrams = Counter([tuple(candidate[i:i+n]) for i in range(len(candidate)-n+1)])
        
        # 裁剪后的匹配数
        clipped = sum(min(count, ref_ngrams[ng]) for ng, count in cand_ngrams.items())
        total = max(len(candidate) - n + 1, 1)
        
        precision = clipped / total if total > 0 else 0
        precisions.append(precision)
    
    # 避免 log(0)
    if any(p == 0 for p in precisions):
        return 0.0
    
    # 几何平均
    log_avg = sum(math.log(p) for p in precisions) / max_n
    
    # 长度惩罚 (Brevity Penalty)
    bp = math.exp(1 - len(reference) / len(candidate)) if len(candidate) < len(reference) else 1.0
    
    return bp * math.exp(log_avg)


# ============================================================
# 3. 对比贪婪解码 vs Beam Search
# ============================================================

print("Beam Search vs 贪婪解码 对比：")
print("=" * 60)

greedy_correct = 0
beam_correct = 0
num_test = 10

for i in range(num_test):
    src_seq, trg_seq = dataset[i]
    src_tensor = torch.LongTensor([src_seq]).to(device)
    
    # 贪婪解码（复用之前的 translate 函数）
    greedy_tokens, _ = translate(encoder, decoder, src_tensor)
    greedy_clean = [t for t in greedy_tokens if t not in [dataset.EOS_token, dataset.PAD_token]]
    
    # Beam Search 解码
    beam_tokens, beam_score = beam_search_decode(encoder, decoder, src_tensor, beam_size=3)
    beam_clean = [t for t in beam_tokens if t not in [dataset.SOS_token, dataset.EOS_token, dataset.PAD_token]]
    
    # 参考答案
    trg_clean = [t for t in trg_seq if t not in [dataset.SOS_token, dataset.EOS_token, dataset.PAD_token]]
    
    # 比较
    greedy_match = greedy_clean == trg_clean
    beam_match = beam_clean == trg_clean
    greedy_correct += greedy_match
    beam_correct += beam_match
    
    # 计算 BLEU
    greedy_bleu = compute_bleu(trg_clean, greedy_clean, max_n=2)
    beam_bleu = compute_bleu(trg_clean, beam_clean, max_n=2)
    
    src_readable = [t-3 for t in src_seq]
    trg_readable = [t-3 for t in trg_clean]
    greedy_readable = [t-3 for t in greedy_clean]
    beam_readable = [t-3 for t in beam_clean]
    
    if i < 5:  # 只打印前 5 条
        g_mark = "✓" if greedy_match else "✗"
        b_mark = "✓" if beam_match else "✗"
        print(f"\n样本 {i+1}:")
        print(f"  源:       {src_readable}")
        print(f"  参考:     {trg_readable}")
        print(f"  贪婪 {g_mark}: {greedy_readable}  BLEU={greedy_bleu:.3f}")
        print(f"  Beam  {b_mark}: {beam_readable}  BLEU={beam_bleu:.3f}")

print(f"\n{'='*60}")
print(f"准确率对比 ({num_test} 条):")
print(f"  贪婪解码:    {greedy_correct}/{num_test} ({greedy_correct/num_test*100:.0f}%)")
print(f"  Beam Search: {beam_correct}/{num_test} ({beam_correct/num_test*100:.0f}%)")
print(f"\n✓ Beam Search 与 BLEU 评估实现完成！")

## 5. 工程实践 (Engineering)

### 关键技术点

1. **Teacher Forcing**：训练时使用真实标签，加速收敛
2. **Padding**：处理变长序列，批量训练
3. **特殊标记**：SOS（开始）、EOS（结束）、PAD（填充）
4. **注意力可视化**：理解模型的对齐关系

### 性能优化

- 使用 GPU 加速训练
- 批量处理提高效率
- 梯度裁剪防止梯度爆炸
- 学习率调度改善收敛

## 6. 综合项目 (Capstone)

本章实现了完整的 Seq2Seq 系统，包括：
- ✅ 编码器-解码器架构
- ✅ 注意力机制
- ✅ 完整训练流程
- ✅ 推理和评估

**实际应用扩展**：
- 使用真实翻译数据集（如 WMT）
- 添加 Beam Search 改善生成质量
- 使用 BPE/WordPiece 处理词汇表
- 实现双向编码器

## 7. 常见问题与调试 (FAQ & Debugging)

### Q1: 为什么需要注意力机制？

**A:** 基础 Seq2Seq 将整个输入压缩成固定长度向量，长序列会丢失信息。注意力机制让解码器能访问所有编码器状态，动态关注相关部分。

### Q2: Teacher Forcing 是什么？

**A:** 训练时使用真实目标序列作为解码器输入，而不是模型预测。这加速训练但可能导致训练/推理不一致。

**解决方案**：
- 使用 Scheduled Sampling（逐渐减少 teacher forcing 比例）
- 在推理时使用 Beam Search

### Q3: 如何处理变长序列？

**A:** 使用 padding 将批次中的序列填充到相同长度，并在损失计算时忽略 padding 标记。

### Q4: 注意力权重总和不等于 1？

**A:** 检查 softmax 是否在正确的维度上应用。注意力权重应该在源序列长度维度上归一化。

### Q5: 模型不收敛怎么办？

**A:** 
- 检查数据预处理是否正确
- 降低学习率
- 使用梯度裁剪
- 增加模型容量（hidden_dim）
- 检查特殊标记（SOS/EOS/PAD）是否正确处理

### 调试技巧

1. **打印形状**：每一步都检查张量形状
2. **可视化注意力**：看模型是否学到正确的对齐
3. **从简单开始**：先在小数据集上验证
4. **监控梯度**：检查是否有梯度消失/爆炸

## 8. 总结与展望 (Summary)

### 核心要点

1. **Seq2Seq 架构**：编码器-解码器结构处理序列到序列任务
2. **注意力机制**：解决信息瓶颈，动态关注相关信息
3. **Teacher Forcing**：训练技巧，加速收敛
4. **实际应用**：机器翻译、文本摘要、对话系统

### 技术演进

```
基础 Seq2Seq (2014)
    ↓
Seq2Seq + Attention (2015)
    ↓
Transformer (2017) ← 纯注意力机制
    ↓
BERT, GPT, T5 等现代模型
```

### 与其他技术的联系

- **RNN/LSTM**：Seq2Seq 的基础组件
- **Attention**：Transformer 的核心
- **Transformer**：下一章的重点

### 下一章预告

**Module 3: Transformer 架构**
- Self-Attention 机制
- Multi-Head Attention
- Position Encoding
- 完整 Transformer 实现

### 💡 思考题

1. 为什么注意力机制能解决长序列问题？
2. Teacher Forcing 的优缺点是什么？
3. 如何改进 Seq2Seq 的生成质量？
4. Seq2Seq 和 Transformer 的主要区别是什么？
5. 如何处理罕见词（OOV）问题？

---

**恭喜！** 你已经掌握了 Seq2Seq 架构和注意力机制的核心概念。

**下一步**：学习 Transformer 架构，理解现代 NLP 模型的基础。

## 思考题参考答案

### 问题 1：为什么注意力机制能解决长序列问题？

**根本原因：消除了固定长度的信息瓶颈。**

**基础 Seq2Seq 的瓶颈：**

在无注意力的 Seq2Seq 中，编码器必须将整个输入序列（无论多长）压缩成**一个**固定维度的向量 $h_T$（最后一个隐藏状态），解码器只能从这一个向量中提取所有信息：

```
x₁, x₂, ..., x₁₀₀  →  [h₁₀₀（单个向量）]  →  y₁, y₂, ...
```

**问题**：
- 信息压缩率极高（100个词压成1个向量），长序列必然丢失早期信息
- 梯度要从 $h_T$ 反向传播到 $h_1$，经过 $T$ 步 LSTM，仍然面临梯度衰减

**注意力机制的解决方案：**

注意力机制让解码器在每个时间步 $t$ 都能直接访问**所有**编码器隐藏状态，并动态计算加权和：

$$c_t = \sum_{i=1}^{T} \alpha_{t,i} h_i, \quad \alpha_{t,i} = \text{softmax}(e_{t,i})$$

```
x₁, x₂, ..., x₁₀₀  →  [h₁, h₂, ..., h₁₀₀（完整保留）]
                                    ↓（动态加权选取）
                         y₁ 用 α₁,   y₂ 用 α₂, ...
```

**优势体现：**

| 方面 | 无注意力 Seq2Seq | 带注意力 Seq2Seq |
|------|----------------|----------------|
| 信息保留 | 仅最后一个隐藏状态 | 所有位置的隐藏状态 |
| 长序列能力 | 急剧下降 | 保持稳定 |
| 梯度路径 | 必须穿越所有时间步 | 可以直接从输出连到对应编码器状态 |
| 可解释性 | 无 | 注意力权重可视化对齐关系 |

**信息论视角**：固定长度向量的信息容量是有限的（约 $d \log_2(d)$ 比特），而注意力机制将信息容量扩展到 $T \times d$（随序列长度线性增长）。

---

### 问题 2：Teacher Forcing 的优缺点是什么？

**Teacher Forcing** 是在训练解码器时，将真实目标序列 $y_{t-1}^*$（而非模型预测 $\hat{y}_{t-1}$）作为当前步的输入。

**优点：**

1. **加速收敛**：
   - 早期训练时模型预测质量很差，若用错误预测作为输入，错误会累积（Exposure Bias 的反面）
   - Teacher Forcing 提供了"正确的起点"，使梯度信号更清晰，训练速度提升显著

2. **训练稳定**：
   - 避免了因早期错误预测导致的"雪崩效应"（一步错、步步错）
   - 损失曲线更平滑，超参数调优更容易

3. **实现简单**：
   - 只需在训练时将 `trg[:, t]` 作为解码器输入，一行代码实现

**缺点：**

1. **训练-推理不一致（Exposure Bias）**：
   - 训练时：解码器总看到真实标签（"完美"输入）
   - 推理时：解码器只能看到自己上一步的预测（可能有误）
   - 这种不一致导致模型在推理时性能下降，尤其是长序列生成

2. **过度依赖真实标签**：
   - 模型学会"依赖"高质量输入，鲁棒性较差
   - 一旦某一步预测出错，后续步骤的质量下降明显

**改进方案：**

| 方案 | 核心思想 | 适用场景 |
|------|---------|---------|
| **Scheduled Sampling** | 逐渐将 teacher forcing 比例从 1 降至 0 | 训练过程逐步过渡 |
| **Beam Search** | 推理时保留 top-k 候选序列，减少单步错误影响 | 推理阶段优化 |
| **REINFORCE** | 用强化学习直接优化序列级别的评价指标（如 BLEU） | 对齐训练目标和评价指标 |
| **Minimum Risk Training** | 最小化期望损失（期望在采样序列上） | 序列级别优化 |

**实践建议**：训练初期使用较高比例的 Teacher Forcing（如 0.8），随训练进行逐渐降低（如线性衰减到 0.3），可以兼顾训练速度和泛化能力。

---

### 问题 3：如何改进 Seq2Seq 的生成质量？

生成质量的提升可以从**解码策略**、**模型架构**和**训练目标**三个层面入手。

**1. 解码策略优化**

- **贪心解码（Greedy Decoding）**：每步选择概率最高的词，简单但次优
- **Beam Search**：保留 $k$ 个最优候选路径，权衡质量与计算量
  ```
  Beam Size = 4: 每步保留 4 条路径，最终取分数最高的序列
  ```
- **Top-k 采样**：从概率最高的 $k$ 个词中随机采样，增加多样性
- **Top-p（Nucleus）采样**：从累积概率超过 $p$ 的最小词集中采样，动态调整候选集大小
- **温度调节**：$p_i^{(T)} = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$，$T < 1$ 使分布更尖锐，$T > 1$ 使分布更平滑

**2. 模型架构改进**

- **多层编码器/解码器**：更深的网络捕捉更复杂的模式（配合残差连接）
- **双向编码器**：使用双向 LSTM 让编码器理解双向上下文
- **Copy Mechanism（指针网络）**：允许模型直接从输入中复制词，有效处理罕见词和专有名词
- **覆盖机制（Coverage Mechanism）**：跟踪已关注的输入位置，避免重复生成或遗漏内容（摘要任务尤为重要）

**3. 训练目标改进**

- **标签平滑（Label Smoothing）**：将目标分布从 one-hot 平滑为 $(1-\epsilon) \cdot \text{one-hot} + \epsilon / V$，防止过度自信
- **序列级训练（MRT/REINFORCE）**：直接优化 BLEU、ROUGE 等序列级评价指标
- **对比学习**：拉近生成序列与参考序列的表示，推远低质量生成

**4. 数据和后处理**

- **数据增强**：回译（Back-translation）生成更多训练样本
- **Subword 分词**：使用 BPE 或 SentencePiece 减少 OOV 问题
- **长度惩罚**：Beam Search 时对短序列加惩罚：$\text{score} = \log p / \text{length}^\alpha$

---

### 问题 4：Seq2Seq 和 Transformer 的主要区别是什么？

Transformer（2017，"Attention is All You Need"）是对 Seq2Seq 架构的根本性革新，摒弃了 RNN/LSTM，完全基于注意力机制。

**核心架构对比：**

| 维度 | Seq2Seq (RNN-based) | Transformer |
|------|---------------------|-------------|
| 基础组件 | LSTM/GRU（循环结构） | Multi-Head Self-Attention |
| 信息传播 | 顺序递推隐藏状态 | 直接计算所有位置对的注意力 |
| 并行化 | 训练时无法并行（顺序依赖） | 训练时完全并行 |
| 长距离依赖 | 需经过多步传播，可能衰减 | 任意两位置一步直达，距离恒为 1 |
| 位置信息 | 天然包含（循环结构隐式编码） | 需要显式位置编码（Positional Encoding） |
| 计算复杂度 | $O(n \cdot d^2)$（串行） | $O(n^2 \cdot d)$（并行，长序列瓶颈） |
| 参数效率 | RNN 参数在所有时间步共享 | 自注意力层参数独立 |

**关键技术差异：**

1. **Self-Attention vs. Cross-Attention**：
   - Seq2Seq：编码器内部无 self-attention，注意力只在解码器中连接到编码器
   - Transformer：编码器和解码器内部都有 self-attention，解码器额外有 cross-attention

2. **Multi-Head Attention**：
   - Transformer 将注意力拆分为 $h$ 个头，每个头学习不同的关注模式
   - 相当于从多个子空间同时理解序列关系

3. **位置编码**：
   - Seq2Seq：位置信息隐含在 LSTM 的时序处理中
   - Transformer：需要显式添加 $\text{PE}(pos, 2i) = \sin(pos / 10000^{2i/d})$

**选择建议：**
- 现代生产系统几乎全面转向 Transformer（及其变体 BERT、GPT、T5）
- Seq2Seq（RNN）仍适用于：计算资源极度受限、需要在线/流式处理的场景

---

### 问题 5：如何处理罕见词（OOV）问题？

**OOV（Out-of-Vocabulary）** 是指测试时出现训练词汇表中未见过的词，会导致模型无法正确处理。

**方法一：子词分词（Subword Tokenization）** — 最主流的方案

将词拆分为更小的单元，使词汇表覆盖几乎所有词：

- **BPE（Byte Pair Encoding）**：
  - 从字符级开始，迭代合并出现频率最高的字符对
  - "unhappiness" → "un", "happi", "ness"
  - GPT 系列使用此方法
  
- **WordPiece**：
  - 类似 BPE，但基于最大化语言模型似然度来合并
  - BERT 使用此方法（前缀 `##` 表示续接）

- **SentencePiece**：
  - 语言无关的实现，支持 BPE 和 Unigram 两种算法
  - 直接在原始字符串上操作（无需预分词），适合中文等无空格语言
  - T5、LLaMA 等使用此方法

**方法二：字符级模型**

直接在字符级别操作，词汇表只需约 256 个字符（ASCII），从根本上消除 OOV：
- 优点：无 OOV，捕捉形态学信息
- 缺点：序列变长，计算量增大，长距离依赖更难建模

**方法三：Copy Mechanism（指针网络）**

对于专有名词、数字等，允许模型直接从输入序列"复制"词：
$$p(\text{word}) = p_{\text{gen}} \cdot p_{\text{vocab}} + (1 - p_{\text{gen}}) \cdot \sum_{i: x_i = \text{word}} \alpha_i$$

其中 $p_{\text{gen}}$ 是"生成"vs"复制"的概率开关。适用于文本摘要、问答等任务。

**方法四：词向量替代**

- 将 OOV 词替换为 `<UNK>` 并赋予特殊的平均词向量
- 使用字符级 CNN/LSTM 动态为 OOV 词生成词向量（如 ELMo 的字符编码层）

**工程实践建议：**

```python
# 推荐：使用 sentencepiece 或 HuggingFace tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
# 词汇表约 21000 个子词，基本无 OOV
```

对于大多数现代 NLP 任务，**子词分词（BPE/WordPiece/SentencePiece）** 已成为事实标准，建议直接采用成熟的 tokenizer 库，而非从零实现。